# Import Environment variables

In [ ]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

In [ ]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

# Import library

In [ ]:
%load_ext autoreload
%autoreload 2
    
import os
import subprocess
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# Data's import

## Data from `50_Breast_Cancer_WITHIN_5Y`

* Femmes n'ayant pas eu de cancer avant l'inclusion (BC and not BC included without time constraints) for all time)
* Origine génétique (RYE)
* Biopsie

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_TEMP_BUCKET')
name_of_file_in_bucket = "df_bc_ko_at_inclusion_223k_nb_BC_WITHIN_5Y_GeneticAncestry_biopsy.tsv"

# save dataframe in a csv file in the same workspace as the notebook
df_bc = pd.read_csv(my_bucket +'/'+ name_of_file_in_bucket, sep=',', low_memory=False)
df_bc

# Family History data processing

## Importing family_history data (statement)

In [ ]:
import pandas
import os

# This query represents dataset "Personal and family health history (with datetime)" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_80480060_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question,
        answer.answer,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (836772)
        )"""

dataset_80480060_survey_df = pandas.read_gbq(
    dataset_80480060_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_80480060_survey_df.head(5)

In [ ]:
dataset_80480060_survey_df['answer'].value_counts()

# Features engineering

## Création d'une colonne par membres de la famille

In [ ]:
list_answer = list(dataset_80480060_survey_df['answer'].unique())
list_answer

In [ ]:
# 1. Nettoyer les noms des colonnes pour ne garder que le membre de la famille
# On retire le préfixe répétitif pour avoir des colonnes lisibles
dataset_80480060_survey_df['clean_answer'] = dataset_80480060_survey_df['answer'].str.split(' - ').str[-1]

In [ ]:
family_list = ['Daughter','Mother','Sibling']

df_survey_family = dataset_80480060_survey_df[dataset_80480060_survey_df['clean_answer'].isin(family_list)]

In [ ]:
df_survey_clean = df_survey_family.copy()

# 2. Création d'une colonne de présence (1) pour le pivot
df_survey_clean['presence'] = 1

# 3. Pivotage : on transforme les valeurs de 'answer' en colonnes
# On utilise 'max' pour que si la personne a répondu 2 fois la même chose, on garde juste 1
df_pivot = df_survey_clean.pivot_table(
    index='person_id', 
    columns='clean_answer', 
    values='presence', 
    aggfunc='max'
).fillna(0).astype(int)

## Traitement des dates

### Ajout d'une colonne `first` et `last`

In [ ]:
dataset_80480060_survey_df['survey_day'] = pd.to_datetime(dataset_80480060_survey_df['survey_datetime'], errors="coerce").dt.tz_localize(None).dt.normalize()

In [ ]:
# 4. Récupération des dates (Première et Dernière réponse)
# C'est important pour savoir à quel moment l'info a été collectée
df_dates = dataset_80480060_survey_df.groupby('person_id')['survey_day'].agg(
    first_survey_date='min',
    last_survey_date='max'
)

### Ajout d'une variable `days_last_survey_to_first_survey` pour identifier le délai entre la 1ère et la dernière réponses au questionnaire

In [ ]:
df_dates['days_last_survey_to_first_survey'] = (df_dates['last_survey_date'] - df_dates['first_survey_date']).dt.days

## Rassemblement des données contenant les antécédent familiaux 

In [ ]:
df_family_history = df_dates.merge(df_pivot, on='person_id', how='right')

# Fusion avec le main set

In [ ]:
df = df_bc.merge(df_family_history, on='person_id', how='left', indicator=True)
df

In [ ]:
df['_merge'].value_counts()

In [ ]:
df = df.drop(columns='_merge')

# Feature engineering - Etape 2

## Ajout d'une variable `days_inclusion_to_first_survey` pour identifier le délai entre la 1ère réponse au questionnaire et la date d'inclusion

In [ ]:
df['inclusion_date'] = pd.to_datetime(df['inclusion_date'], errors="coerce").dt.tz_localize(None).dt.normalize()

In [ ]:
df['days_inclusion_to_first_survey'] = (df['first_survey_date'] - df['inclusion_date']).dt.days

## Ajout d'une variable `days_inclusion_to_last_survey` pour identifier le délai entre la dernière réponse au questionnaire et la date d'inclusion

In [ ]:
df['days_inclusion_to_last_survey'] = (df['last_survey_date'] - df['inclusion_date']).dt.days

## Ajout d'une variable `days_has_bc_to_first_survey` pour obtenir le délai entre la date de cancer du sein et la date de questionnaire

In [ ]:
df['first_breast_cancer_date'] = pd.to_datetime(df['first_breast_cancer_date'], errors="coerce").dt.tz_localize(None).dt.normalize().dropna()

In [ ]:
df['days_has_bc_to_first_survey'] = (df['first_breast_cancer_date'] - df['first_survey_date']).dt.days.dropna()

## Ajout d'une variable `days_has_bc_to_last_survey` pour obtenir le délai entre la date de cancer du sein et la date de questionnaire

In [ ]:
df['days_has_bc_to_last_survey'] = (df['first_breast_cancer_date'] - df['last_survey_date']).dt.days.dropna()

### Nombre d'individus déjà déjà un cancer du sein en répondant au questionnaire

In [ ]:
len(df[df['days_has_bc_to_last_survey'] < 0]) # Déjà malades à l'inclusion

### Nombre d'individus ayant eu un cancer après le questionnaire, soit pendant l'étude

In [ ]:
len(df[df['days_has_bc_to_last_survey'] > 0]) # Devenus malades pendant l'étude

# EDA

In [ ]:
df_family_history = df[['person_id','first_survey_date','last_survey_date','days_last_survey_to_first_survey','Daughter','Mother','Sibling','days_inclusion_to_first_survey','days_inclusion_to_last_survey']].dropna()

## EDA - Univarié

In [ ]:
# Création de l'histogramme avec échelle log sur l'axe Y
cols = ['days_last_survey_to_first_survey','days_inclusion_to_first_survey','days_inclusion_to_last_survey']

for col in cols:
    plt.figure(figsize=(10, 4))
    df_family_history[col].hist(bins=50, log=True, color='black')
    
    plt.title(f'Distribution de {col} (Échelle Log)')
    plt.xlabel(col)
    plt.ylabel('Nombre de patients (Log)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Création de l'histogramme avec échelle log sur l'axe Y
cols = ['days_last_survey_to_first_survey','days_inclusion_to_first_survey','days_inclusion_to_last_survey']

df_multi_survey_answer = df_family_history.loc[df['days_inclusion_to_first_survey'] != df['days_inclusion_to_last_survey']]

for col in cols:
    plt.figure(figsize=(10, 4))
    df_multi_survey_answer[col].hist(bins=50, log=True, color='black')
    
    plt.title(f'Distribution de {col} (Échelle Log)\n Pour les individus ayant répondu à plusieurs questionnaires')
    plt.xlabel(col)
    plt.ylabel('Nombre de patients (Log)')
    plt.tight_layout()
    plt.show()

# Combinations analysis 

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
from upsetplot import UpSet

# 1️⃣ Masquer les avertissements pour nettoyer la console
warnings.simplefilter(action="ignore", category=FutureWarning)

# 2️⃣ Extraction dynamique des colonnes d'antécédents
cols = df_family_history[['Daughter','Mother','Sibling']].columns.tolist()

# Correction de la variable N pour correspondre à ton DataFrame
N = len(df_family_history)

# 3️⃣ Pré-agrégation avec Pandas (indispensable pour contourner le bug NumPy 2.x)
df_bool = df_family_history[cols].astype(bool)
data_counts = df_bool.groupby(cols).size()

# 4️⃣ Configuration de la figure
fig = plt.figure(figsize=(14, 10))

# 5️⃣ Création de l'UpSetPlot
# /!\ On retire show_counts et show_percentages pour éviter le crash de type TypeError
up = UpSet(data_counts, sort_by="cardinality", element_size=45)

up.plot(fig=fig)

# 6️⃣ Titre principal (ajusté avec 'y' pour éviter les chevauchements)
plt.suptitle(
    f"Intersection des antécédents familiaux\n"
    f"Distribution selon le type de parenté (N = {N})\n",
    fontsize=12,
    fontweight="bold",
)

plt.show()

In [ ]:
# 1️⃣ On masque les alertes pour la propreté du notebook
warnings.simplefilter(action="ignore", category=FutureWarning)

# 2️⃣ Ton filtrage : on ne garde que les individus avec un cancer du sein
df_filtered = df[df["has_bc"] == 1]

# 3️⃣ Pré-agrégation Pandas (indispensable pour contourner le crash de NumPy)
df_bool = df_filtered[cols].dropna().astype(bool)
data_counts = df_bool.groupby(cols).size()

# Le N correspond maintenant uniquement aux personnes filtrées
BC = len(df_filtered)
N = len(df_bool)

# 4️⃣ Configuration de la figure
fig = plt.figure(figsize=(14, 10))

# 5️⃣ Création de l'UpSetPlot
# /!\ On supprime show_counts et show_percentages pour éviter le TypeError
up = UpSet(data_counts, sort_by="cardinality", element_size=45)

up.plot(fig=fig)

# 6️⃣ Titre
plt.suptitle(
    f"Intersection des antécédents familiaux pour les individus ayant eu un cancer du sein\n"
    f"Distribution selon le type de parenté (BC = {BC}; N = {N})",
    fontsize=12,
    fontweight="bold",
)

plt.show()

In [ ]:
summary = (
    df
    .groupby(cols)["has_bc"]
    .agg(
        n_individuals="count",
        n_bc="sum",
        prevalence_bc="mean"
    )
    .reset_index()
)

summary["prevalence_bc"] = summary["prevalence_bc"] * 100

summary

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=summary,
    x="prevalence_bc",
    y=summary[cols].astype(str).agg(" + ".join, axis=1),
    color="#B43E3B"
)

plt.xlabel("Prévalence du cancer du sein (%)")
plt.ylabel("Combinaison des antécédents familiaux")
plt.title("Prévalence du cancer selon les intersection des antécédents familiaux\nDaughter + Mother + Sibling", weight="bold")

plt.show()

# Data's export

In [ ]:
destination_filename = 'Datas/df_bc_ko_at_inclusion_223k_nb_BC_WITHIN_5Y_GeneticAncestry_biopsy_FamilyHistory.tsv'
df.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_TEMP_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr